# Optional: learned proposals and physical verification

NeRD learns an approximation of trajectories generated by the physical solver. It is unnecessary for a direct sweep. After training, we freeze the network weights and differentiate a cost with respect to design parameters. A different cost changes the search, not the physics the network learned.

The current expanded adapter assumes a supported, zero-phase start and fixed 200 V / 363 Hz actuation. Its broad-domain accuracy checks failed. This notebook examines recorded evidence and does not train an AI model by default.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the repository or tutorials directory."
os.chdir(ROOT)
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/walking-robot-mpl")
print("Repository:", ROOT)
print("Python:", sys.executable)


In [ ]:
import yaml
configuration = yaml.safe_load(Path("examples/designs/fixed_actuation_1m_s.yaml").read_text())
print("Documented physical design:", configuration["design"])
TRAINING_REPORT = Path("outputs/expanded_nerd/training/report.json")
if TRAINING_REPORT.exists():
    training = json.loads(TRAINING_REPORT.read_text())
    print("Selected neural iteration:", training["selected_iteration"])
    print("Held-out test gate passed:", training["test"]["trajectory_gate_passed"])
else:
    print("No local trained model. Read docs/expanded_design.md or use the optional training commands below.")


In [ ]:
def speed_cost(mean_speed_m_s, target_m_s):
    return (mean_speed_m_s - target_m_s) ** 2

VERIFICATION_REPORT = Path("outputs/expanded_nerd/training/physical_verification/report.json")
if VERIFICATION_REPORT.exists():
    proposals = json.loads(VERIFICATION_REPORT.read_text())["proposals"]
    for row in proposals:
        proposal = row["proposal"]
        print({"target_m_s": proposal["target_m_s"], "predicted_m_s": proposal["neural_speed_m_s"], "solver_m_s": row["physical_speed_m_s"], "forecast_passed": row["neural_forecast_passed"], "target_met": row["target_met"]})
else:
    print("No local proposal report. Predictions must be generated and checked before accepting a design.")


A prediction of 1.069 m/s produced only 0.572 m/s with the original solver. That proposal was rejected; it did not replace the separate accepted 1.005 m/s physical design. Physical plausibility alone is insufficient: a plausible motion can still have the wrong speed.

A useful research cycle samples a broad domain, simulates data, trains NeRD, optimizes a cost, verifies proposals and adds large-error cases to training. Independent validation and test designs must remain reserved. More random designs may help coverage, but transitions and sparse regions can still cause errors. Full three-variable gradient validation remains outstanding.


In [ ]:
RUN_TRAINING = False  # Advanced, optional and substantially more expensive than reading this notebook.
NERD_ROOT = Path("../neural-robot-dynamics")
commands = [
    [sys.executable, "tools/generate_expanded_data.py"],
    [sys.executable, "tools/generate_expanded_data.py", "--collect"],
    [sys.executable, "tools/train_expanded_nerd.py", "--nerd-root", str(NERD_ROOT)],
    [sys.executable, "tools/verify_expanded_nerd.py"],
    [sys.executable, "tools/verify_expanded_nerd.py", "--collect"],
]
if RUN_TRAINING:
    assert NERD_ROOT.exists(), "Install the official NeRD checkout and optional learning dependencies first."
    for command in commands:
        subprocess.run(command, check=True)
else:
    print("Training skipped. Read docs/expanded_design.md for dependencies, output directories and exact validation criteria.")
